# Batched vertex-patch preconditioner (`precond=vsbatch`) on an A100

The 3D channel solve with the condensed vertex-patch Schwarz preconditioner:
71 CG iterations per stage against Jacobi's 4675 (BALANCED_CONDENSED_PLAN.md).
Its apply is dense batched fp64 triangular solves, which the Mac CPU does at
~70 GFLOP/s (70 s per step) and the Apple GPU cannot do at all
(FP64_ON_APPLE_GPU.md).  This notebook measures the same code on an A100:
fp64 everywhere, torch backend for the operator, `LSSEM3D_DEVICE=cuda` for the
patch solves.

**Runtime → Change runtime type → A100 GPU** first.  Colab does not always give
you the A100; cell 1 checks.

In [ ]:
# 1. Which GPU did we get?
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import subprocess
name = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', name)
if 'A100' not in name:
    print('!! not an A100 -- fp64 numbers from this device are not comparable (L4/T4 fp64 is 1/32-1/64 of fp32)')

In [ ]:
# 2. Get the code (public repo, branch main).  Idempotent: clone once, pull afterwards.
import os
if os.path.isdir('/content/lssem/.git'):
    !cd /content/lssem && git fetch -q origin main && git reset -q --hard origin/main
else:
    !git clone -q --branch main https://github.com/chandc/Python_SEM.git /content/lssem
%cd /content/lssem
!git log --oneline -1
!pip install -q numba scipy matplotlib ninja     # ninja: the fused cuda backend compiles its extension with it
import torch, numpy, scipy, numba
print('torch', torch.__version__, torch.cuda.is_available(), '| numpy', numpy.__version__, '| scipy', scipy.__version__, '| numba', numba.__version__)

In [ ]:
# 3. Smoke test of the whole path on a tiny mesh (~2 min): a failure here is a setup problem, not a performance one.
!python -u colab/run_vsbatch_a100.py --quick --out /content/results_quick --backends torch

In [ ]:
# 4. The real measurement on the production channel mesh (6x18 N=8, 17 modes).
#    Builds three preconditioners (one per RKW3 stage), then times Jacobi and vsbatch steps
#    on the torch backend and on the fused 'cuda' backend (compiled on first use, ~1-2 min).
#    Mac M3 Max CPU reference, fp64: apply 0.21 s/iteration, step 70 s (vsbatch) / ~80 s (Jacobi).
#    GB10 reference (fused cuda backend): Jacobi 60 s/step, vsbatch 60 s/step (apply 271 ms, bandwidth-bound).
!python -u colab/run_vsbatch_a100.py --out /content/results_a100 --backends torch,cuda

## Optional: the correctness gate against run01

The two-step restart needs run01's last checkpoint (`checkpoint_0006200.npz`,
24 MB, not in git).  Put it in Google Drive under `MyDrive/lssem_data/` (or
upload it with the file picker) and run the next cell.  The vsbatch step must
reproduce the Jacobi line to every printed digit:

    t=4.961 u_tau=0.9937 U_b=15.840 rms_w=0.9201 E=897.30 eps=103.54 div=8.64e-04 conv=0.000 CFL=1.11   (CG 4675 Jacobi, ~72 vsbatch)

In [ ]:
# 5. Restart gate (optional).  Finds run01's checkpoint_0006200.npz in this order:
#    (a) an environment variable CKPT_URL (a direct download link, e.g. a GitHub release asset),
#    (b) Google Drive: MyDrive/lssem_data/checkpoint_0006200.npz,
#    (c) the file picker.
import os, urllib.request
ck = None
url = os.environ.get('CKPT_URL', '')          # e.g. https://github.com/chandc/Python_SEM/releases/download/<tag>/checkpoint_0006200.npz
if url:
    ck = '/content/checkpoint_0006200.npz'; urllib.request.urlretrieve(url, ck)
if ck is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        cand = '/content/drive/MyDrive/lssem_data/checkpoint_0006200.npz'
        ck = cand if os.path.exists(cand) else None
    except Exception as e:
        print('Drive not mounted:', e)
if ck is None:
    from google.colab import files
    up = files.upload()          # pick checkpoint_0006200.npz (24 MB)
    ck = next(iter(up)) if up else None
print('checkpoint:', ck)
if ck:
    !python -u colab/run_vsbatch_a100.py --out /content/results_a100 --backends torch --ckpt {ck}

In [ ]:
# 6. Keep the results (Colab VMs are ephemeral).
!cat /content/results_a100/summary.md 2>/dev/null || cat /content/results_quick/summary.md
try:
    from google.colab import drive; drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/lssem_results && cp -r /content/results_* /content/drive/MyDrive/lssem_results/ && echo saved to Drive
except Exception as e:
    print('not saved to Drive:', e)